In [1]:
import pyogrio

# List layers
layers = pyogrio.list_layers("input/bushfire_extent.gdb")
print(layers)


[['NT_Historical_Bushfire_Extents_v1' 'MultiPolygon']
 ['National_Historical_Bushfire_Extents_v4' 'MultiPolygon']]


In [3]:
# Get metadata only (instant, no data loaded)
info = pyogrio.read_info("input/bushfire_extent.gdb", layer="National_Historical_Bushfire_Extents_v4")
print(info['features'])  # row count
print(info['fields'])  # column names


345345
['fire_id' 'fire_name' 'ignition_date' 'capture_date' 'extinguish_date'
 'fire_type' 'ignition_cause' 'capt_method' 'area_ha' 'perim_km' 'state'
 'agency' 'SHAPE_Length' 'SHAPE_Area']


In [4]:
# Load with simplified geometry (much faster)
gdf = pyogrio.read_dataframe("input/bushfire_extent.gdb",
                              layer="National_Historical_Bushfire_Extents_v4",
                              columns=["fire_id", "area_ha", "state", "ignition_date"])
# ^ loading only these columns + geometry

# Extract centroids
gdf["lon"] = gdf.geometry.centroid.x
gdf["lat"] = gdf.geometry.centroid.y

# Drop geometry — just keep the flat data
df = gdf.drop(columns="geometry")

print(df.head())

/Users/MasaTemma/Desktop/FIT5120-TE04-Main-Project/.venv/lib/python3.13/site-packages/pyogrio/raw.py:200: RuntimeWarning: organizePolygons() received a polygon with more than 100 parts.  The processing may be really slow.  You can skip the processing by setting METHOD=SKIP.
  return ogr_read(
/var/folders/w4/l4h21lrj0d59d23sfzvj3t5m0000gn/T/ipykernel_22098/3737720532.py:8: UserWarning: Geometry is in a geographic CRS. Results from 'centroid' are likely incorrect. Use 'GeoSeries.to_crs()' to re-project geometries to a projected CRS before this operation.

  gdf["lon"] = gdf.geometry.centroid.x
/var/folders/w4/l4h21lrj0d59d23sfzvj3t5m0000gn/T/ipykernel_22098/3737720532.py:9: UserWarning: Geometry is in a geographic CRS. Results from 'centroid' are likely incorrect. Use 'GeoSeries.to_crs()' to re-project geometries to a projected CRS before this operation.

  gdf["lat"] = gdf.geometry.centroid.y


  fire_id             ignition_date  area_ha  \
0   FB511                       NaT      0.0   
1   FB892                       NaT     14.0   
2     NaN 1920-01-01 00:00:00+00:00  43205.0   
3     NaN 1926-01-01 00:00:00+00:00  42551.0   
4     NaN 1931-01-01 00:00:00+00:00    705.0   

                                state         lon        lat  
0  ACT (Australian Capital Territory)  149.069006 -35.220220  
1  ACT (Australian Capital Territory)  149.076242 -35.440667  
2  ACT (Australian Capital Territory)  148.888895 -35.522131  
3  ACT (Australian Capital Territory)  148.846916 -35.411622  
4  ACT (Australian Capital Territory)  148.827200 -35.385858  


In [ ]:
# Extract centroids
gdf["lon"] = gdf.geometry.centroid.x
gdf["lat"] = gdf.geometry.centroid.y

# Drop geometry — just keep the flat data
df = gdf.drop(columns="geometry")

print(df.head())

In [5]:
df.to_csv("input/bushfire_centroids.csv", index=False)

In [6]:
import pyogrio
import pandas as pd

# Basic info
info = pyogrio.read_info("input/bushfire_extent.gdb", layer="National_Historical_Bushfire_Extents_v4")
print(info)

# Load without geometry for stats
df = pyogrio.read_dataframe("input/bushfire_extent.gdb", 
                             layer="National_Historical_Bushfire_Extents_v4",
                             read_geometry=False)

print(df.dtypes)
print(df.describe())
print(df['state'].value_counts())
print(df['ignition_date'].min(), df['ignition_date'].max())  # date range

{'layer_name': 'National_Historical_Bushfire_Extents_v4', 'crs': 'EPSG:4283', 'encoding': 'UTF-8', 'fields': array(['fire_id', 'fire_name', 'ignition_date', 'capture_date',
       'extinguish_date', 'fire_type', 'ignition_cause', 'capt_method',
       'area_ha', 'perim_km', 'state', 'agency', 'SHAPE_Length',
       'SHAPE_Area'], dtype=object), 'dtypes': array(['object', 'object', 'datetime64[ms]', 'object', 'datetime64[ms]',
       'object', 'object', 'object', 'float64', 'float64', 'object',
       'object', 'float64', 'float64'], dtype=object), 'ogr_types': ['OFTString', 'OFTString', 'OFTDateTime', 'OFTString', 'OFTDateTime', 'OFTString', 'OFTString', 'OFTString', 'OFTReal', 'OFTReal', 'OFTString', 'OFTString', 'OFTReal', 'OFTReal'], 'ogr_subtypes': ['OFSTNone', 'OFSTNone', 'OFSTNone', 'OFSTNone', 'OFSTNone', 'OFSTNone', 'OFSTNone', 'OFSTNone', 'OFSTNone', 'OFSTNone', 'OFSTNone', 'OFSTNone', 'OFSTNone', 'OFSTNone'], 'fid_column': 'OBJECTID', 'geometry_name': 'SHAPE', 'geometry_type'